<a href="https://colab.research.google.com/github/kasrasa/Object-detection-tutorial/blob/YOLO/YOLO_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U pycocotools
!pip install -q -U ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 4.1 MB/s eta 0:00:00


In [2]:
import os
import random
import shutil
import urllib.request
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.ops import box_iou
from torchvision.transforms.functional import to_tensor
from ultralytics import YOLO

from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from pycocotools.coco import COCO
from IPython.display import display

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Torch version: 2.11.0+cpu
CUDA available: False


In [3]:
# -------------------------
# Global configuration
# -------------------------

SEED = 42
DEVICE = 0 if torch.cuda.is_available() else "cpu"  # Ultralytics accepts GPU index or "cpu"
TORCH_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# COCO annotation paths uploaded in Colab.
TRAIN_ANN = "/content/data/train/instances_train2014.json"
VAL_ANN = "/content/data/valid/instances_val2014.json"
TRAIN_INSTANCES_ROOT = Path("/content/data/train/")
VAL_INSTANCES_ROOT = Path("/content/data/valid/")
TRAIN_INSTANCES_ROOT.mkdir(parents=True, exist_ok=True)
VAL_INSTANCES_ROOT.mkdir(parents=True, exist_ok=True)

# Image roots. Images are downloaded on demand into these folders.
TRAIN_IMAGE_ROOT = Path("/content/data/images/train2014")
VAL_IMAGE_ROOT = Path("/content/data/images/val2014")
TRAIN_IMAGE_ROOT.mkdir(parents=True, exist_ok=True)
VAL_IMAGE_ROOT.mkdir(parents=True, exist_ok=True)

# YOLO-native exported dataset root.
YOLO_DATA_ROOT = Path("/content/data/yolo_coco_small")
YOLO_ORIGINAL_ROOT = YOLO_DATA_ROOT / "original"
YOLO_EXPANDED_ROOT = YOLO_DATA_ROOT / "expanded"

# Dataset sizes.
NUM_TRAIN = 200
NUM_VAL = 50
MIN_SMALL_OBJECTS_PER_IMAGE = 3
NUM_ADDED_HARD_IMAGES = 100

# YOLO model and training settings.
# Change to "yolo11n.pt" or "yolov8n.pt" if you want an older baseline.
YOLO_WEIGHTS = "yolo26n.pt"
IMG_SIZE = 640
BATCH_SIZE = 8
NUM_WORKERS = 4
NUM_EPOCHS = 10
PATIENCE = 0
FREEZE_LAYERS = None  # Example: 10 to freeze early layers. None means no freeze argument.

# Evaluation / mining settings.
IOU_THRESH = 0.5 # regular iou threshold to match detected bbs to ground truth
SCORE_THRESH = 0.05 # intentionally low to see if model can detect objects with low confidence or completely misses them
POOR_RECALL_THRESHOLD = 0.7 # used to find weak classes and candidate classes for hard mining
MIN_SMALL_GT = 3 # number of small gt objects in the image that is not ocluded or crowded

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Using Ultralytics device:", DEVICE)
print("Train annotations exist:", os.path.exists(TRAIN_ANN))
print("Val annotations exist:", os.path.exists(VAL_ANN))

Using Ultralytics device: cpu
Train annotations exist: False
Val annotations exist: False


In [4]:
coco_train = COCO(TRAIN_ANN)
coco_val = COCO(VAL_ANN)

loading annotations into memory...


FileNotFoundError: [Errno 2] No such file or directory: '/content/data/train/instances_train2014.json'

In [ ]:
def build_coco_yolo_category_maps(coco):
  coco_to_yolo = {}
  yolo_to_coco = {}
  class_names = []
  cat_ids = sorted(coco.getCatIds())
  categories = coco.loadCats(cat_ids)
  for idx, cat in enumerate(categories):
    coco_to_yolo[cat["id"]] = idx # coco ids mapped to yolo
    yolo_to_coco[idx] = cat["id"] # yolo ids mapped to coco
    class_names.append(cat["name"])
  return coco_to_yolo, yolo_to_coco, class_names

def validate_ann(ann):
  if ann["iscrowd"] == 1:
    return False

  x, y, w, h = ann["bbox"]
  if ann.get("area", w*h) <= 1:
    return False
  if ann["bbox"][2] <= 1 or ann["bbox"][3] <= 1:
    return False
  return True

def coco_ann_to_yolo_row(ann, img_w, img_h, coco_to_yolo):
    x, y, w, h = ann["bbox"]

    cx = (x + w / 2) / img_w
    cy = (y + h / 2) / img_h
    bw = w / img_w
    bh = h / img_h

    class_id = coco_to_yolo[ann["category_id"]]

    return [class_id, cx, cy, bw, bh]

def create_yolo_dataset(anns, coco, coco_to_yolo):
  yolo_dataset = defaultdict(list)

  for ann in anns:
    image_id = ann["image_id"]
    image_info = coco.loadImgs(image_id)[0]

    img_w, img_h = image_info["width"], image_info["height"]
    yolo_row = coco_ann_to_yolo_row(ann, img_w, img_h, coco_to_yolo)
    yolo_dataset[image_id].append(yolo_row)
  return yolo_dataset

coco_to_yolo, yolo_to_coco, class_names = build_coco_yolo_category_maps(coco_train)
anns = coco_train.loadAnns(coco_train.getAnnIds())
yolo_dataset = create_yolo_dataset(anns, coco_train, coco_to_yolo)
print(list(yolo_dataset.items())[:5])